# Kinyarwanda News Topic Classification

Classifying Kinyarwanda news articles into 14 topics (politics, sport, economy, health,
entertainment, history, technology, tourism, culture, fashion, religion, environment,
education, relationship) on the **KINNEWS** benchmark (Niyongabo et al., 2020).

| | |
|---|---|
| **Open in Colab** | `[[TODO: Colab badge URL — needs the GitHub repo to exist first]]` |
| **Live app (Hugging Face Space)** | `[[TODO: Space URL — Phase 7]]` |
| **Model on the Hub** | `[[TODO: Hub model URL — Phase 7]]` |
| **Video walkthrough** | `[[TODO: video link]]` |

### What this notebook does

It compares four families of text classifier on the same data and the same splits:

1. **Sparse linear models** — TF-IDF word n-grams and character n-grams + Logistic Regression.
2. **A recurrent model** — a BiLSTM written from scratch in PyTorch, with and without
   fastText subword embeddings trained on the training split.
3. **Multilingual Transformers** — XLM-R base, fine-tuned.
4. **African-centric Transformers** — AfroXLMR base, fine-tuned.

It then runs two ablations, a full evaluation with confidence intervals and significance
tests, a detailed error analysis, and deploys the winner as a Gradio app.

### How to read this notebook

Every code cell is preceded by a markdown cell saying **what** it does, **why** it is done
that way, and **what alternative was rejected**. Every number reported anywhere in this
project is written by code into `results/` and read back from there — no figure in the
report or README is typed by hand.

---
**Build status:** Section 0 is complete (Phase 1). Sections 1–13 are scaffolded and are
filled in by later build phases; each carries a visible `[[TODO: ...]]` marker until then.

## 0. Setup

### 0.1 Environment detection and dependency install

**What this does.** Works out whether we are on Google Colab or a local machine, and on
Colab installs the exact pinned versions from `requirements.txt` so the notebook behaves
the same in both places.

**Why.** The project is developed locally on CPU but the Transformer runs (E3, E4) need a
GPU, which means Colab. Pinning the same versions in both environments is what makes a
result reproducible rather than "it worked on my machine that day".

**Alternative rejected.** Letting Colab use its own preinstalled `transformers`/`torch`.
Rejected because Colab upgrades those images without warning, so a notebook that ran in
September can silently behave differently in October — exactly the kind of drift that
invalidates a reported number.

In [ ]:
import os
import sys
import platform
import subprocess

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

# Pinned versions. Kept identical to requirements.txt at the repo root.
PINNED = [
    "torch==2.14.0",
    "transformers==4.57.6",
    "datasets==4.8.5",
    "accelerate==1.15.0",
    "gensim==4.4.0",
    "scikit-learn==1.8.0",
    "jupytext==1.19.5",
]

if IN_COLAB:
    print("Running on Google Colab -> installing pinned dependencies (quiet, ~2 min).")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *PINNED],
        check=True,
    )
else:
    print("Running locally -> assuming the pinned virtualenv is already active.")

print(f"python   : {platform.python_version()}")
print(f"platform : {platform.platform()}")
print(f"in_colab : {IN_COLAB}")

### 0.2 The `CONFIG` dictionary

**What this does.** Defines every tunable value used anywhere in the project — data
choices, preprocessing rules, model hyperparameters, evaluation settings — in one place.

**Why one dict.** Two reasons. First, it is the single thing that has to be copied into a
results file for a run to be reproducible, so every `results/runs/*.json` embeds a copy of
it. Second, it means a hyperparameter can never disagree between the place it is set and
the place it is reported; there is only one place.

**Alternative rejected.** Scattering hyperparameters at the point of use (e.g. writing
`lr=2e-5` inside the training call). That reads more naturally line by line, but it makes
the notebook impossible to audit — you cannot answer "what learning rate produced this
number?" without re-reading every cell.

**On `fast_mode`.** When `True`, the data is subsampled and training is cut to a couple of
steps so the whole notebook runs end to end in under ~10 minutes on a CPU. This is a
smoke test that the code path works — **its numbers are meaningless and are never
reported**. Result files written in fast mode are tagged so they can never be mistaken for
real ones.

In [ ]:
CONFIG = {
    # ---------- identity ----------
    "project": "kinyarwanda-news-classification",
    "seed": 42,
    # fast_mode: subsample + tiny training so the notebook smoke-tests quickly on CPU.
    # Set to False for every number that is actually reported.
    "fast_mode": True,

    # ---------- data ----------
    "data": {
        "hf_id": "kinnews_kirnews",
        # The raw (not pre-cleaned) version: Transformers expect natural text, and this
        # project documents its own preprocessing rather than inheriting someone else's.
        "hf_config": "kinnews_raw",
        "num_labels": 14,
        "val_size": 0.10,          # stratified slice of the official train split
        "label_column": "label",
        "title_column": "title",
        "content_column": "content",
    },

    # ---------- quality audit ----------
    "audit": {
        "near_dup_threshold": 0.90,   # TF-IDF cosine above this = near-duplicate
        "min_words": 10,              # articles shorter than this are flagged
        "label_spot_check_n": 20,     # articles shown for manual review by the student
    },

    # ---------- preprocessing ----------
    "preprocess": {
        # "title_content" = title + " . " + content (default), "title_only" = ablation A1
        "text_mode": "title_content",
        "url_token": "<url>",
        "email_token": "<email>",
        "num_token": "<num>",
    },

    # ---------- B1 / E1: sparse linear models ----------
    "classical": {
        "word_ngram_range": (1, 2),
        "word_min_df": 2,
        "word_max_features": 200_000,
        "char_ngram_range": (2, 5),
        "char_analyzer": "char_wb",
        "char_min_df": 3,
        "char_max_features": 300_000,
        "sublinear_tf": True,
        "C_grid": [0.1, 1.0, 10.0],   # tuned on validation only
        "class_weight": "balanced",
        "max_iter": 2000,
        "top_features_per_class": 15,
    },

    # ---------- E2: BiLSTM ----------
    "bilstm": {
        "embed_dim": 300,
        "hidden_dim": 128,            # per direction; concatenated -> 256
        "num_layers": 1,
        "bidirectional": True,
        "dropout": 0.3,
        "max_tokens": 400,
        "min_freq": 2,                # vocabulary built on the train split only
        "lr": 1e-3,
        "batch_size": 64,
        "epochs": 15,
        "patience": 3,                # early stopping on validation macro-F1
        "grad_clip": 1.0,
        "class_weighted_loss": True,
    },

    # ---------- E2b: fastText embeddings ----------
    "fasttext": {
        "sg": 1,                      # skip-gram (Bojanowski et al., 2017)
        "vector_size": 300,
        "window": 5,
        "min_count": 2,
        "epochs": 10,
        "negative": 5,
        "min_n": 3,                   # character n-gram range for subword vectors
        "max_n": 6,
        "workers": 4,
        "neighbour_probe_words": [],  # filled in Phase 4 with real Kinyarwanda words
    },

    # ---------- E3 / E4: Transformers ----------
    "transformer": {
        "models": {
            "E3": "xlm-roberta-base",
            "E4": "Davlan/afro-xlmr-base",
        },
        "lr": 2e-5,
        "warmup_ratio": 0.1,
        "weight_decay": 0.01,
        "lr_scheduler_type": "linear",
        "epochs": 3,
        "max_length": 512,
        "per_device_batch_size": 8,
        "grad_accum_steps": 2,        # effective batch 16
        "fp16": True,                 # GPU only; disabled automatically on CPU
        "metric_for_best_model": "eval_macro_f1",
    },

    # ---------- ablations ----------
    "ablations": {
        "A1_text_mode": "title_only",
        "A2_max_length": 256,
    },

    # ---------- evaluation ----------
    "eval": {
        "primary_metric": "macro_f1",
        "bootstrap_n": 1000,
        "bootstrap_ci": 0.95,
        "n_confident_mistakes": 10,
        "n_taxonomy_errors": 30,
        "length_buckets": [0, 100, 250, 500, 10**9],
    },
}

# fast_mode overrides: applied here so the rest of the notebook never branches on the flag.
if CONFIG["fast_mode"]:
    CONFIG["fast"] = {
        "n_train": 800,
        "n_val": 200,
        "n_test": 200,
    }
    CONFIG["classical"]["C_grid"] = [1.0]
    CONFIG["bilstm"]["epochs"] = 2
    CONFIG["bilstm"]["max_tokens"] = 128
    CONFIG["fasttext"]["epochs"] = 2
    CONFIG["transformer"]["epochs"] = 1
    CONFIG["transformer"]["max_length"] = 128
    CONFIG["eval"]["bootstrap_n"] = 100
    CONFIG["ablations"]["A2_max_length"] = 64

print("fast_mode:", CONFIG["fast_mode"])
if CONFIG["fast_mode"]:
    print("  !! Numbers produced in this mode are a smoke test only and are never reported.")
print(f"seed: {CONFIG['seed']}   primary metric: {CONFIG['eval']['primary_metric']}")

### 0.3 Project paths (and Google Drive on Colab)

**What this does.** Finds the repository root, creates the output folders, and on Colab
mounts Google Drive so checkpoints and result files survive a disconnected session.

**Why Drive.** Colab reclaims its VM without warning. A fine-tuning run that takes 40
minutes and writes only to `/content` is lost when that happens. Writing results to Drive,
plus the "skip if the result JSON already exists" rule used by every experiment, means a
dropped session resumes instead of restarting.

**Alternative rejected.** Re-downloading and re-running everything each session. Rejected
on time: the full experiment suite is several GPU-hours, and Colab sessions do not
reliably last that long.

In [ ]:
from pathlib import Path


def find_project_root() -> Path:
    """Walk up from the current directory looking for a repo marker."""
    markers = ("README.md", "requirements.txt", ".git")
    for start in (Path.cwd(), Path.cwd().parent):
        for candidate in [start, *start.parents]:
            if any((candidate / m).exists() for m in markers):
                return candidate
    return Path.cwd()


DRIVE_ROOT = None
if IN_COLAB:
    try:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = Path("/content/drive/MyDrive") / CONFIG["project"]
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        PROJECT_ROOT = DRIVE_ROOT
        print(f"Drive mounted. Artefacts persist at: {DRIVE_ROOT}")
    except Exception as exc:  # not fatal: the notebook still runs, it just will not resume
        print(f"Drive mount failed ({exc}); falling back to ephemeral /content storage.")
        PROJECT_ROOT = Path("/content") / CONFIG["project"]
else:
    PROJECT_ROOT = find_project_root()

PATHS = {
    "root": PROJECT_ROOT,
    "data": PROJECT_ROOT / "data",
    "audit": PROJECT_ROOT / "data" / "audit",
    "results": PROJECT_ROOT / "results",
    "runs": PROJECT_ROOT / "results" / "runs",
    "predictions": PROJECT_ROOT / "results" / "predictions",
    "figures": PROJECT_ROOT / "figures",
    "report": PROJECT_ROOT / "report_materials",
    "models": PROJECT_ROOT / "models",
    "hf_cache": PROJECT_ROOT / "data" / "hf_cache",
}
for _name, _path in PATHS.items():
    _path.mkdir(parents=True, exist_ok=True)

CONFIG["paths"] = {k: str(v) for k, v in PATHS.items()}

print(f"project root: {PATHS['root']}")
for _name in ("data", "results", "figures", "report", "models"):
    print(f"  {_name:10} -> {PATHS[_name]}")

### 0.4 Seeding

**What this does.** Fixes the random seed for Python, NumPy and PyTorch, and asks cuDNN
for deterministic kernels.

**Why.** Shuffling, the validation split, dropout masks and weight initialisation are all
random. Without a fixed seed, re-running the notebook gives a slightly different macro-F1
and there is no way to tell a real improvement from noise.

**Honest caveat, which belongs in the defense.** This makes a run *repeatable*, not
*robust*. A single seed still does not tell us whether E4 beating E3 by a point is a real
effect or a lucky initialisation — multi-seed training is explicitly out of scope here
(Section 11), which is why the bootstrap confidence intervals and McNemar's test in
Section 8 carry that argument instead.

**Alternative rejected.** Averaging over 3–5 seeds per model. That is the statistically
right answer, and it was rejected purely on compute budget: it would multiply the
Transformer runs, which dominate the cost, by 3–5.

In [ ]:
import random

import numpy as np


def set_seed(seed: int) -> None:
    """Seed every source of randomness the project touches."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except ImportError:
        pass


set_seed(CONFIG["seed"])
print(f"seeded everything with {CONFIG['seed']}")

### 0.5 Hardware check

**What this does.** Reports the device the notebook will train on, and turns off `fp16`
when there is no GPU.

**Why it is printed.** Training time and the feasibility of E3/E4 depend entirely on this.
`fp16` (half precision) is a GPU feature; leaving it on for a CPU run raises an error, so
the flag is derived from the hardware rather than trusted from `CONFIG`.

In [ ]:
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CONFIG["transformer"]["fp16"] = bool(torch.cuda.is_available()) and CONFIG["transformer"]["fp16"]

print(f"torch        : {torch.__version__}")
print(f"device       : {DEVICE}")
if torch.cuda.is_available():
    _props = torch.cuda.get_device_properties(0)
    print(f"gpu          : {_props.name} ({_props.total_memory / 1e9:.1f} GB)")
else:
    print(f"cpu threads  : {torch.get_num_threads()}")
    print("  No GPU: E3/E4 full fine-tuning is not practical here. Use fast_mode locally")
    print("  for a smoke test and run the real Transformer experiments on Colab.")
print(f"fp16 enabled : {CONFIG['transformer']['fp16']}")

## 1. Introduction and research questions

`[[TODO: Phase 2 — written alongside the dataset section]]`

The four questions every experiment in this notebook exists to answer:

| RQ | Question |
|---|---|
| RQ1 | How far can a simple sparse model go, and do character n-grams beat word n-grams? |
| RQ2 | Do subword embeddings trained on in-domain text help a BiLSTM over random init? |
| RQ3 | Does African-centric pretraining (AfroXLMR) beat general multilingual (XLM-R)? |
| RQ4 | What does the best model still get wrong, and why? |

## 2. Related work notes

`[[TODO: Phase 8 — short, cited; mirrors report_materials/related_work_notes.md]]`

## 3. Dataset

`[[TODO: Phase 2 — loading, EDA, quality audit, preprocessing, splits]]`

## 4. Baselines (B0, B1)

`[[TODO: Phase 3]]`

## 5. Classical and recurrent experiments (E1, E2a, E2b)

`[[TODO: Phase 3 for E1, Phase 4 for E2a/E2b]]`

## 6. Transformer experiments (E3, E4)

`[[TODO: Phase 5]]`

## 7. Ablations (A1, A2)

`[[TODO: Phase 5]]`

## 8. Evaluation of the final model

`[[TODO: Phase 6]]`

## 9. Error analysis

`[[TODO: Phase 6]]`

## 10. Deployment

`[[TODO: Phase 7]]`

## 11. Limitations and future work

`[[TODO: Phase 8]]`

## 12. Conclusion

`[[TODO: Phase 8 — one to three sentences per RQ, with real numbers]]`

## 13. References

`[[TODO: Phase 8 — mirrors report_materials/references.bib, every entry verified]]`